In [ ]:
#FINAL MODEL THreshold COVERAGE 
import pandas as pd
models = [
    ("MBERT", 0.7660, r"Projects\ADHD2\analysis_results\MBERT\best\MBERT_best_similarity_matrix_clipped.csv"),
    ("LaBSE", 0.5221, r"Projects\ADHD2\analysis_results\LaBSE\best\LaBSE_best_similarity_matrix_clipped.csv"),
    ("mE5L", 0.8776, r"Projects\ADHD2\analysis_results\mE5L\best\mE5L_best_similarity_matrix_clipped.csv"),
]

for name, thr, path in models:
    df = pd.read_csv(path)
    if not df.columns[0].startswith("B_"):
        df = df.set_index(df.columns[0])
    best_per_b = df.max(axis=0)
    covered = (best_per_b >= thr).sum()
    total = len(best_per_b)
    print(f"{name}: {covered}/{total} ({covered/total*100:.2f}%)")


In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from pathlib import Path
from kneed import KneeLocator

# =========================
# 0. 설정
# =========================

model_name = "MBERT"
thr = 0.7660

sim_path = Path(r"Projects\ADHD2\analysis_results\MBERT\best\MBERT_best_similarity_matrix_clipped.csv")
a_path = Path(r"Projects\ADHD2\Data\Aset.csv")
b_path = Path(r"Projects\ADHD2\Data\Bset.csv")

output_xlsx = Path(r"Projects\ADHD2\analysis_results\MBERT_covercheck_kneedle.xlsx")
output_fig = Path(r"Projects\ADHD2\analysis_results\MBERT_kneedle_coverage_curve.png")

# =========================
# 1. 데이터 로드
# =========================

aset = pd.read_csv(a_path, encoding="utf-8-sig").copy()
bset = pd.read_csv(b_path, encoding="utf-8-sig").copy()

if "faq_id" not in aset.columns:
    aset.insert(0, "faq_id", [f"A_{i}" for i in range(len(aset))])

if "b_id" not in bset.columns:
    bset.insert(0, "b_id", [f"B_{i}" for i in range(len(bset))])

sim = pd.read_csv(sim_path, encoding="utf-8-sig")

if not sim.columns[0].startswith("B_"):
    sim = sim.set_index(sim.columns[0])

sim = sim.clip(lower=0)

print(f"Model: {model_name}")
print(f"Threshold: {thr}")
print(f"Similarity matrix shape: {sim.shape}")

# =========================
# 2. 각 B 질문의 best-match A 산출
# =========================

best_sim = sim.max(axis=0)
best_a = sim.idxmax(axis=0)

best_match_df = pd.DataFrame({
    "b_id": best_sim.index,
    "best_a_id": best_a.values,
    "best_similarity": best_sim.values,
})

best_match_df["covered"] = best_match_df["best_similarity"] >= thr

total_b = len(best_match_df)
covered_b_n = int(best_match_df["covered"].sum())
uncovered_b_n = total_b - covered_b_n

print({
    "total_b": total_b,
    "covered_b_n": covered_b_n,
    "uncovered_b_n": uncovered_b_n,
})

# =========================
# 3. threshold 이상 B를 커버한 A별 count 계산
# =========================

covered_best_match_df = best_match_df[best_match_df["covered"]].copy()

covered_counts = (
    covered_best_match_df["best_a_id"]
    .value_counts()
    .sort_values(ascending=False)
)

coverage_curve_df = pd.DataFrame({
    "rank": np.arange(1, len(covered_counts) + 1),
    "faq_id": covered_counts.index,
    "covered_B_count": covered_counts.values,
})

coverage_curve_df["cumulative_covered_B_count"] = (
    coverage_curve_df["covered_B_count"].cumsum()
)

coverage_curve_df["cumulative_coverage_pct_total_B"] = (
    coverage_curve_df["cumulative_covered_B_count"] / total_b * 100
)

coverage_curve_df["cumulative_coverage_pct_covered_B"] = (
    coverage_curve_df["cumulative_covered_B_count"] / covered_b_n * 100
)

coverage_curve_df["incremental_coverage_pct_total_B"] = (
    coverage_curve_df["covered_B_count"] / total_b * 100
)

# =========================
# 4. Kneedle로 knee k 탐색
# =========================

x = coverage_curve_df["rank"].values
y = coverage_curve_df["cumulative_coverage_pct_total_B"].values

kl = KneeLocator(
    x,
    y,
    curve="concave",
    direction="increasing",
    S=1.0,
)

if kl.knee is not None:
    knee_k = int(kl.knee)
    knee_coverage_pct = float(kl.knee_y)
    kneedle_status = "found"
else:
    knee_k = len(coverage_curve_df)
    knee_coverage_pct = float(
        coverage_curve_df["cumulative_coverage_pct_total_B"].iloc[-1]
    )
    kneedle_status = "not_found_used_all"

print({
    "kneedle_status": kneedle_status,
    "knee_k": knee_k,
    "coverage_at_knee_pct_total_B": knee_coverage_pct,
})

# =========================
# 5. Knee k까지의 A 질문 목록 생성
# =========================

knee_top_a_ids = coverage_curve_df.head(knee_k)["faq_id"].tolist()

knee_top_a_df = coverage_curve_df.head(knee_k).copy()

knee_top_a_df = knee_top_a_df.merge(
    aset[["faq_id", "QuestionA", "AnswerA"]],
    on="faq_id",
    how="left",
)

knee_top_a_df = knee_top_a_df[[
    "rank",
    "faq_id",
    "QuestionA",
    "AnswerA",
    "covered_B_count",
    "incremental_coverage_pct_total_B",
    "cumulative_covered_B_count",
    "cumulative_coverage_pct_total_B",
    "cumulative_coverage_pct_covered_B",
]]

# =========================
# 6. Bset에 best-match A 정보 붙이기
# =========================

a_question_map = aset.set_index("faq_id")["QuestionA"].to_dict()

b_result_df = bset.merge(
    best_match_df,
    on="b_id",
    how="left",
)

b_result_df["best_a_question"] = b_result_df["best_a_id"].map(a_question_map)

if "question_full" not in b_result_df.columns:
    raise ValueError("Bset.csv에 question_full 컬럼이 없습니다.")

covered_b_df = b_result_df[b_result_df["covered"]].copy()
uncovered_b_df = b_result_df[~b_result_df["covered"]].copy()

covered_b_df = covered_b_df[[
    "b_id",
    "question_full",
    "best_a_id",
    "best_a_question",
    "best_similarity",
    "covered",
]]

uncovered_b_df = uncovered_b_df[[
    "b_id",
    "question_full",
    "best_a_id",
    "best_a_question",
    "best_similarity",
    "covered",
]]

# =========================
# 6-1. 커버하지 못한 A 질문 목록 생성
# =========================

covered_a_ids = set(covered_counts.index)
all_a_ids = set(aset["faq_id"])

uncovered_a_ids = sorted(
    all_a_ids - covered_a_ids,
    key=lambda value: int(str(value).replace("A_", ""))
)

best_b_for_a = sim.idxmax(axis=1)
best_sim_for_a = sim.max(axis=1)

b_question_map = bset.set_index("b_id")["question_full"].to_dict()

uncovered_a_df = pd.DataFrame({
    "faq_id": uncovered_a_ids,
})

uncovered_a_df["covered_B_count"] = 0

uncovered_a_df = uncovered_a_df.merge(
    aset[["faq_id", "QuestionA", "AnswerA"]],
    on="faq_id",
    how="left",
)

uncovered_a_df["best_b_id"] = uncovered_a_df["faq_id"].map(best_b_for_a)
uncovered_a_df["best_b_question"] = uncovered_a_df["best_b_id"].map(b_question_map)
uncovered_a_df["best_similarity_to_B"] = uncovered_a_df["faq_id"].map(best_sim_for_a)

uncovered_a_df = uncovered_a_df[[
    "faq_id",
    "QuestionA",
    "AnswerA",
    "covered_B_count",
    "best_b_id",
    "best_b_question",
    "best_similarity_to_B",
]]

# =========================
# 7. Metrics 생성
# =========================

final_coverage_pct = covered_b_n / total_b * 100

coverage_at_knee_pct_covered_B = (
    knee_top_a_df["cumulative_coverage_pct_covered_B"].iloc[-1]
    if len(knee_top_a_df) > 0
    else np.nan
)

covered_a_n = len(covered_counts)
total_a = len(aset)

coverage_ratio_total_B = (
    knee_coverage_pct / final_coverage_pct
    if final_coverage_pct != 0
    else np.nan
)

faq_ratio_covered_A = (
    knee_k / covered_a_n
    if covered_a_n != 0
    else np.nan
)

faq_ratio_total_A = (
    knee_k / total_a
    if total_a != 0
    else np.nan
)

slope = coverage_curve_df["cumulative_coverage_pct_total_B"].diff()
slope = slope.fillna(
    coverage_curve_df["cumulative_coverage_pct_total_B"].iloc[0]
)

slope_init = float(slope.iloc[0]) if len(slope) > 0 else np.nan
slope_at_knee = float(slope.iloc[knee_k - 1]) if knee_k > 0 else np.nan

slope_ratio = (
    slope_at_knee / slope_init
    if slope_init != 0
    else np.nan
)

curvature = slope.diff().fillna(0)
curvature_at_knee = float(curvature.iloc[knee_k - 1]) if knee_k > 0 else np.nan

metrics_df = pd.DataFrame([{
    "model": model_name,
    "threshold": thr,
    "kneedle_status": kneedle_status,
    "total_A_count": total_a,
    "total_B_count": total_b,
    "covered_B_count": covered_b_n,
    "uncovered_B_count": uncovered_b_n,
    "covered_A_count": covered_a_n,
    "uncovered_A_count": len(uncovered_a_df),
    "knee_k_A_count": knee_k,
    "coverage_at_knee_pct_total_B": knee_coverage_pct,
    "coverage_at_knee_pct_covered_B": coverage_at_knee_pct_covered_B,
    "final_coverage_pct_total_B": final_coverage_pct,
    "coverage_ratio_knee_to_final": coverage_ratio_total_B,
    "faq_ratio_knee_to_covered_A": faq_ratio_covered_A,
    "faq_ratio_knee_to_total_A": faq_ratio_total_A,
    "slope_init": slope_init,
    "slope_at_knee": slope_at_knee,
    "slope_ratio": slope_ratio,
    "curvature_at_knee": curvature_at_knee,
}])

print(metrics_df.T)

# =========================
# 8. Kneedle coverage curve
# =========================

plot_df = coverage_curve_df.copy()

label_top_n = min(30, len(plot_df))

fig, ax1 = plt.subplots(figsize=(14, 7))

bars = ax1.bar(
    plot_df["rank"],
    plot_df["incremental_coverage_pct_total_B"],
    color="lightsteelblue",
    alpha=0.65,
    label="Incremental coverage by each FAQ",
)

ax1.set_xlabel("FAQ rank")
ax1.set_ylabel("Incremental coverage (%)", color="steelblue")
ax1.tick_params(axis="y", labelcolor="steelblue")
ax1.set_xlim(0, min(len(plot_df) + 1, max(knee_k + 10, label_top_n + 2)))

max_bar = plot_df["incremental_coverage_pct_total_B"].max()

ax1.set_ylim(0, 25)
ax1.set_yticks([0, 5, 10, 15, 20, 25])

for _, row in plot_df.head(label_top_n).iterrows():
    ax1.text(
        row["rank"],
        row["incremental_coverage_pct_total_B"] + 0.3,
        f'{row["incremental_coverage_pct_total_B"]:.1f}%',
        ha="center",
        va="bottom",
        fontsize=8,
        rotation=90,
        color="steelblue",
        clip_on=True,
    )

ax2 = ax1.twinx()

ax2.step(
    plot_df["rank"],
    plot_df["cumulative_coverage_pct_total_B"],
    where="post",
    color="royalblue",
    linewidth=2.5,
    label="Cumulative B coverage",
)

ax2.scatter(
    [knee_k],
    [knee_coverage_pct],
    color="crimson",
    s=100,
    zorder=5,
    label=f"Knee: k={knee_k}, cumulative={knee_coverage_pct:.2f}%",
)

ax2.vlines(
    knee_k,
    ymin=0,
    ymax=knee_coverage_pct,
    colors="crimson",
    linestyles="dashed",
    alpha=0.7,
)

ax2.hlines(
    knee_coverage_pct,
    xmin=0,
    xmax=knee_k,
    colors="crimson",
    linestyles="dotted",
    alpha=0.7,
)

ax2.set_ylabel("Cumulative coverage (%)", color="royalblue")
ax2.tick_params(axis="y", labelcolor="royalblue")

ax2.set_ylim(
    0,
    max(100, plot_df["cumulative_coverage_pct_total_B"].max() * 1.05)
)

plt.title(
    f"Coverage Curve with Per-FAQ Incremental Coverage "
    f"({model_name}, threshold={thr})"
)

handles1, labels1 = ax1.get_legend_handles_labels()
handles2, labels2 = ax2.get_legend_handles_labels()

ax1.legend(
    handles1 + handles2,
    labels1 + labels2,
    loc="upper right",
    fontsize=9,
)

ax1.grid(axis="y", alpha=0.25)

fig.tight_layout(rect=[0, 0, 1, 0.96])

output_fig.parent.mkdir(parents=True, exist_ok=True)
plt.savefig(output_fig, dpi=300)
plt.show()

print(f"Figure saved: {output_fig}")

output_xlsx.parent.mkdir(parents=True, exist_ok=True)

with pd.ExcelWriter(output_xlsx, engine="openpyxl") as writer:
    knee_top_a_df.to_excel(writer, index=False, sheet_name="knee_top_A")
    covered_b_df.to_excel(writer, index=False, sheet_name="covered_B")
    uncovered_b_df.to_excel(writer, index=False, sheet_name="uncovered_B")
    uncovered_a_df.to_excel(writer, index=False, sheet_name="uncovered_A")
    coverage_curve_df.to_excel(writer, index=False, sheet_name="coverage_curve")
    metrics_df.to_excel(writer, index=False, sheet_name="metrics")

print({
    "output_excel": str(output_xlsx),
    "output_figure": str(output_fig),
    "knee_k": knee_k,
    "covered_B_count": covered_b_n,
    "uncovered_B_count": uncovered_b_n,
    "covered_A_count": covered_a_n,
    "uncovered_A_count": len(uncovered_a_df),
})